In [6]:
!pip install transformers datasets evaluate -q

import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [7]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [8]:
# Shuffle and take a small subset for fast training
small_train = dataset["train"].shuffle(seed=42).select(range(800))
small_test = dataset["test"].shuffle(seed=42).select(range(200))

print("Train examples:", len(small_train))
print("Test examples:", len(small_test))
print("\nSample review:", small_train[0]["text"][:300])
print("\nLabel:", small_train[0]["label"], "(0 = negative, 1 = positive)")

Train examples: 800
Test examples: 200

Sample review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot 

Label: 1 (0 = negative, 1 = positive)


In [9]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Try it on one example sentence
sample_text = "This movie was great"
tokens = tokenizer(sample_text)
print(tokens)

print("\nDecoded back:", tokenizer.convert_ids_to_tokens(tokens["input_ids"]))

{'input_ids': [101, 2023, 3185, 2001, 2307, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

Decoded back: ['[CLS]', 'this', 'movie', 'was', 'great', '[SEP]']


In [10]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",   # pad every review to the same length
        truncation=True,        # cut off reviews that are too long
        max_length=256          # keep first 256 tokens - plenty for sentiment, keeps training fast
    )

train_tokenized = small_train.map(tokenize_function, batched=True)
test_tokenized = small_test.map(tokenize_function, batched=True)

print(train_tokenized)
print("\nFirst tokenized example length:", len(train_tokenized[0]["input_ids"]))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 800
})

First tokenized example length: 256


In [11]:
# Remove columns the model doesn't need, keep only what's required for training
train_tokenized = train_tokenized.remove_columns(["text"])
test_tokenized = test_tokenized.remove_columns(["text"])

train_tokenized = train_tokenized.rename_column("label", "labels")  # Trainer expects "labels", not "label"
test_tokenized = test_tokenized.rename_column("label", "labels")

train_tokenized.set_format("torch")
test_tokenized.set_format("torch")

print(train_tokenized)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 800
})


In [28]:
!pip install nbformat -q

In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # positive / negative
)
model = model.to(device if 'device' in dir() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

print(model.config)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.10.2",
  "vocab_size": 30522
}



In [13]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=20,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

print("Trainer ready")

Trainer ready


In [15]:
!pip install -U datasets -q

In [16]:
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.537530,0.453382,0.805000
2,0.326966,0.358263,0.830000
3,0.187040,0.346164,0.835000


TrainOutput(global_step=150, training_loss=0.3729445473353068, metrics={'train_runtime': 55.5648, 'train_samples_per_second': 43.193, 'train_steps_per_second': 2.7, 'total_flos': 158960878387200.0, 'train_loss': 0.3729445473353068, 'epoch': 3.0})

In [17]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [18]:
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.107418,0.486228,0.845000
2,0.088540,0.494525,0.840000
3,0.027127,0.459827,0.860000


TrainOutput(global_step=150, training_loss=0.1034453038374583, metrics={'train_runtime': 59.2077, 'train_samples_per_second': 40.535, 'train_steps_per_second': 2.533, 'total_flos': 158960878387200.0, 'train_loss': 0.1034453038374583, 'epoch': 3.0})

In [19]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
model = model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print("Model loaded")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [20]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1).max().item()

    label = "Positive" if prediction == 1 else "Negative"
    print(f"Text: {text}")
    print(f"Prediction: {label} (confidence: {confidence:.2%})")

predict_sentiment("This was the best movie I've seen all year, absolutely loved it.")
predict_sentiment("Waste of time, terrible acting and a boring plot.")
predict_sentiment("It was okay, nothing special but not bad either.")

Text: This was the best movie I've seen all year, absolutely loved it.
Prediction: Negative (confidence: 55.31%)
Text: Waste of time, terrible acting and a boring plot.
Prediction: Negative (confidence: 55.85%)
Text: It was okay, nothing special but not bad either.
Prediction: Negative (confidence: 55.92%)


In [21]:
print(trainer.state.global_step)
print(trainer.state.log_history[-3:] if trainer.state.log_history else "No training history found")

150
[{'loss': 0.027127256989479064, 'grad_norm': 0.1924564093351364, 'learning_rate': 1.4666666666666669e-06, 'epoch': 2.8, 'step': 140}, {'eval_loss': 0.4598274230957031, 'eval_accuracy': 0.86, 'eval_runtime': 1.5258, 'eval_samples_per_second': 131.08, 'eval_steps_per_second': 8.52, 'epoch': 3.0, 'step': 150}, {'train_runtime': 59.2077, 'train_samples_per_second': 40.535, 'train_steps_per_second': 2.533, 'total_flos': 158960878387200.0, 'train_loss': 0.1034453038374583, 'epoch': 3.0, 'step': 150}]


In [22]:
print("Model device:", next(model.parameters()).device)
print("Model training mode?:", model.training)

Model device: cuda:0
Model training mode?: False


In [23]:
device = next(model.parameters()).device

def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1).max().item()

    label = "Positive" if prediction == 1 else "Negative"
    print(f"Text: {text}")
    print(f"Prediction: {label} (confidence: {confidence:.2%})")

predict_sentiment("This was the best movie I've seen all year, absolutely loved it.")
predict_sentiment("Waste of time, terrible acting and a boring plot.")
predict_sentiment("It was okay, nothing special but not bad either.")

Text: This was the best movie I've seen all year, absolutely loved it.
Prediction: Negative (confidence: 55.31%)
Text: Waste of time, terrible acting and a boring plot.
Prediction: Negative (confidence: 55.85%)
Text: It was okay, nothing special but not bad either.
Prediction: Negative (confidence: 55.92%)


In [24]:
text = "This was the best movie I've seen all year, absolutely loved it."
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

print("Raw logits:", outputs.logits)
print("Softmax probabilities:", torch.softmax(outputs.logits, dim=-1))

Raw logits: tensor([[ 0.1604, -0.0526]], device='cuda:0')
Softmax probabilities: tensor([[0.5531, 0.4469]], device='cuda:0')


In [25]:
print("Is trainer.model the same object as model?", trainer.model is model)

Is trainer.model the same object as model? False


In [26]:
model = trainer.model
model.eval()

device = next(model.parameters()).device

predict_sentiment("This was the best movie I've seen all year, absolutely loved it.")
predict_sentiment("Waste of time, terrible acting and a boring plot.")
predict_sentiment("It was okay, nothing special but not bad either.")

Text: This was the best movie I've seen all year, absolutely loved it.
Prediction: Positive (confidence: 98.73%)
Text: Waste of time, terrible acting and a boring plot.
Prediction: Negative (confidence: 98.95%)
Text: It was okay, nothing special but not bad either.
Prediction: Positive (confidence: 59.02%)


In [27]:
model.save_pretrained("sentiment_distilbert")
tokenizer.save_pretrained("sentiment_distilbert")

!zip -r sentiment_distilbert.zip sentiment_distilbert

from google.colab import files
files.download("sentiment_distilbert.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: sentiment_distilbert/ (stored 0%)
  adding: sentiment_distilbert/model.safetensors (deflated 8%)
  adding: sentiment_distilbert/config.json (deflated 50%)
  adding: sentiment_distilbert/tokenizer_config.json (deflated 43%)
  adding: sentiment_distilbert/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Phase 1 — Deep Learning Foundations

Three end-to-end deep learning projects built from scratch in PyTorch and Hugging Face, covering the three core architecture families: CNN (spatial data), LSTM (sequential/time-series data), and Transformer (pretrained fine-tuning).

---

## Project 1: DocSense — CNN Document Classifier

Classifies scanned business documents (invoice, letter, email) from visual layout alone, using a CNN built from scratch.

- **Dataset:** RVL-CDIP subset, 480 images, 3 classes
- **Result:** 70.83% test accuracy
- **Key finding:** Initial model overfit heavily (97% train vs 74% test). Applying dropout + data augmentation reduced the train/test gap from 23.2 points to 11.7 points — a more trustworthy, generalizable model, even at similar headline accuracy.
- **Confusion matrix insight:** Invoices were most often misclassified as letters, traced to low table-density invoice scans that visually resemble plain text documents.

[Full details in subfolder README]

---

## Project 2: Sales Forecasting — LSTM Time Series

Forecasts future sales from historical monthly data using an LSTM, tested on both real and synthetic retail data.

- **Datasets:** (1) Real-world Shampoo Sales dataset (36 months), (2) synthetic Coca-Cola-style data with engineered seasonality (3 years, daily, resampled to monthly)
- **Results:**
  - Real data: LSTM MAE 82.60 vs naive baseline MAE 159.78 (**48% improvement**)
  - Synthetic data: LSTM MAE 80.06 vs naive baseline MAE 92.21 (**13% improvement**)
- **Key finding:** The LSTM provided much greater value on noisy, volatile real-world data than on smooth, strongly seasonal synthetic data — because the naive baseline already performs reasonably well when data is smooth. This demonstrates why benchmarking against a naive baseline is essential: raw accuracy metrics alone can be misleading.

[Full details in subfolder README]

---

## Project 3: Sentiment Classifier — Fine-tuned DistilBERT

Fine-tuned a pretrained DistilBERT transformer for binary sentiment classification (positive/negative movie reviews).

- **Dataset:** IMDB reviews, 800 training / 200 test examples
- **Result:** 86% test accuracy, achieved in 3 epochs (~1 minute of training)
- **Key finding:** With a fraction of the data used by the CNN project, fine-tuning a pretrained transformer dramatically outperformed training a CNN from scratch — demonstrating the practical value of transfer learning when labeled data is limited.
- **Debugging note:** Diagnosed and fixed a silent model-reference bug during development, where a stale `model` variable pointed to a freshly re-initialized (untrained) model instead of the actual trained weights held by the `Trainer` object. Diagnosed by inspecting raw logits directly rather than trusting wrapper code, and resolved by reassigning `model = trainer.model`.

[Full details in subfolder README]

---

## Cross-project takeaways

1. **Always compare against a naive/simple baseline** — a model's raw accuracy number means little without that context (see LSTM project).
2. **A smaller train/test gap is more valuable than a higher raw accuracy** if it means better generalization to unseen data (see CNN project).
3. **Pretrained models dramatically reduce the data and time needed** for strong performance compared to training from scratch (see Transformer project vs CNN project).
4. **Verify the actual object in memory, not just the code path** — silent reference bugs (like `trainer.model` vs `model`) can produce misleading results that look like a model failure but are actually a debugging issue.

## Tech stack

`Python` · `PyTorch` · `torchvision` · `Hugging Face Transformers` · `Hugging Face Datasets` · `scikit-learn` · `evaluate` · `matplotlib` / `seaborn` · Google Colab (T4 GPU)

## How to run

Each project has its own Colab notebook:
- `01_cnn_invoice_classifier.ipynb`
- `02_lstm_sales_forecast.ipynb`
- `03_transformer_sentiment_classifier.ipynb`

Open in Google Colab, set Runtime → T4 GPU, and run all cells in order.